# CURE-Rec — full reproducible workflow

This notebook runs the complete Milestone 1 workflow in the correct evidence order:

1. fetch or load an external recommendation dataset;
2. standardize and audit its available evidence;
3. profile the data and run every registered CPU baseline model;
4. run the CURE-Sim oracle causal game across all 64 intervention coalitions and configured scenarios;
5. select a direct robust-improvement portfolio;
6. inspect structured logs, numbered tables, figures, manifests, and decision assets.

**Evidence discipline:** external ratings data are useful for loading, model logic, and baseline analysis. CURE-Sim remains the complete causal/oracle benchmark until an audited slate-policy log and sequential OPE estimator are available.

## 1. Setup

Run from `paper-ideas/CURE-Rec/code/` after `python -m pip install -e '.[dev]'`. The root detection also supports launching Jupyter from the repository root.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the CURE-Rec code directory or repository root.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cure_rec.analysis import analyze_dataset
from cure_rec.config import load_settings
from cure_rec.data import audit_interactions, load_curesim, load_dataset, write_standardized_dataset
from cure_rec.pipeline import run_experiment

print('Project root:', ROOT)


## 2. Configure the causal benchmark run

Use `quick` for an interactive laptop pass and `full` for the larger CURE-Sim configuration. Both run all six interventions, exact 64-coalition games, all configured scenarios, robust selection, and the complete numbered asset contract.

In [ ]:
CURE_MODE = 'quick'  # quick | full
config_name = 'curesim_quickstart.yaml' if CURE_MODE == 'quick' else 'curesim_full.yaml'
config_path = ROOT / 'configs' / config_name
settings = load_settings(config_path)

print('Config:', config_name)
print('Config hash:', settings.config_hash())
print('Users / items / horizon:', settings.simulator.n_users, settings.simulator.n_items, settings.simulator.horizon)
print('Interventions:', list(settings.interventions.costs))
print('Scenarios:', [scenario.name for scenario in settings.scenarios])

# Explicit CURE-Sim data loading.
simulator = load_curesim(settings)
print('CURE-Sim catalogue shape:', simulator.catalog.features.shape)
print('CURE-Sim public-profile shape:', simulator.state.public_profiles.shape)


## 3. Fetch or load public/local interaction data

The default fetches MovieLens-1M only after explicit consent in this cell. Available choices are `movielens_1m`, `coat`, `yahoo_r3`, and `csv`. Yahoo! R3 must be downloaded manually under its access terms.

The loader standardizes interactions and then audits whether the dataset supports only descriptive analysis, semi-synthetic analysis, short-horizon OPE, or sensitivity-bounded policy claims.

In [ ]:
PUBLIC_DATASET = 'movielens_1m'  # movielens_1m | coat | yahoo_r3 | csv
PUBLIC_SOURCE = ROOT / 'data' / 'raw' / PUBLIC_DATASET
FETCH_IF_MISSING = True  # explicit network consent for MovieLens-1M or Coat
STANDARDIZED_OUTPUT = ROOT / 'data' / 'processed' / f'{PUBLIC_DATASET}_interactions.csv'

# For DATASET='csv', set PUBLIC_SOURCE to the CSV file itself.
public_result = load_dataset(PUBLIC_DATASET, PUBLIC_SOURCE, download=FETCH_IF_MISSING)
public_audit = audit_interactions(public_result.interactions)

print('Dataset:', public_result.dataset)
print('Rows:', len(public_result.interactions))
print('Permitted claim:', public_audit.permitted_claim)
print('Audit notes:', *public_audit.notes, sep='\n- ')
display(public_result.interactions.head())

# Persist only after inspecting the audit; this does not change the claim level.
write_standardized_dataset(public_result, STANDARDIZED_OUTPUT)
print('Standardized interactions:', STANDARDIZED_OUTPUT)


## 4. Data logic and baseline-model analysis

This step profiles the loaded data and runs all registered CPU baselines:

- popularity ranking;
- NumPy BPR matrix factorization when chronological positives are available.

The outputs are external-data model-analysis assets. They are not presented as causal intervention results.

In [ ]:
RUN_BPR = True
BPR_UPDATES = 50_000 if CURE_MODE == 'quick' else 200_000
MAX_EVAL_USERS = 1_000

data_analysis = analyze_dataset(
    public_result,
    output_root=ROOT / 'runs',
    run_bpr=RUN_BPR,
    bpr_updates=BPR_UPDATES,
    max_eval_users=MAX_EVAL_USERS,
    seed=settings.run.seed,
)

print('External-data analysis run:', data_analysis.run_dir)
print('Evidence level:', data_analysis.audit.permitted_claim)
display(data_analysis.summary)
display(data_analysis.model_metrics)


## 5. Run the full CURE-Rec causal workflow

This executes every coalition of the six intervention players under every configured CURE-Sim scenario, computes exact Shapley values and Grabisch–Roubens interactions, performs direct robust-improvement selection with abstention, and generates the full numbered paper-asset contract.

In [ ]:
logger, game, decision = run_experiment(settings)
RUN_DIR = logger.run_dir

print('CURE-Rec run:', RUN_DIR)
print('Decision:', decision.action)
print('Selected portfolio:', decision.selected_interventions)
print('Worst-case improvement:', round(decision.lower_improvement, 5))


## 6. Inspect attribution, interactions, and direct robust selection

The decision rule uses direct worst-case coalition improvement. Shapley regions explain and audit that decision; lower Shapley endpoints are never summed to choose a portfolio.

In [ ]:
display(game.regions.sort_values('phi_mean', ascending=False))
display(game.interaction_table.sort_values('interaction_mean', ascending=False))

coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
    lower_improvement=('improvement', 'min'),
    upper_improvement=('improvement', 'max'),
    cost=('cost', 'first'),
    interventions=('active_interventions', 'first'),
).sort_values('lower_improvement', ascending=False)
display(coalitions.head(12))


## 7. Inspect all generated paper assets and structured logs

The asset registry records every generated table/figure together with any manual or future evidence dependency. The JSONL event stream provides the detailed behind-the-scenes coalition evaluation timeline.

In [ ]:
asset_manifest = json.loads((RUN_DIR / 'artifacts' / 'asset_manifest.json').read_text())
asset_registry = pd.DataFrame(asset_manifest)
display(asset_registry)

print('Generated tables:')
for path in sorted((RUN_DIR / 'tables').glob('*.csv')):
    print('-', path.name)
print('\nGenerated figures:')
for path in sorted((RUN_DIR / 'figures').glob('*.png')):
    print('-', path.name)

events_path = RUN_DIR / 'logs' / 'events.jsonl'
events = pd.DataFrame([json.loads(line) for line in events_path.read_text().splitlines()])
display(events[['timestamp_utc', 'event']].tail(20))

card = json.loads((RUN_DIR / 'artifacts' / 'explanation_card.json').read_text())
card


## 8. Re-run from the command line

The same end-to-end units are available outside Jupyter:

```bash
cure-rec load-data --dataset movielens_1m --source data/raw/movielens_1m --download
cure-rec analyze-data --dataset movielens_1m --source data/raw/movielens_1m
cure-rec simulate --config configs/curesim_full.yaml
```

The `full-run` command orchestrates external-data loading, auditing, baseline analysis, and the CURE-Sim causal asset run in one command.